# SCD Type 0: No Change 
SCD Type 0 is best for immutable records where we do not want original records to be updated and kept intact. For this we simply check what is new and insert it and ignore everything else.  

In [0]:
%sql
CREATE OR REPLACE TABLE sandbox.bronze.dim_calendar (
	id BIGINT GENERATED ALWAYS AS IDENTITY,
	calendar_date DATE,
	calendar_year INT,
	fiscal_year INT,
	fiscal_quarter INT,
	month_number INT,
	month_name STRING,
	day_of_month INT,
	day_of_week_number INT,
	day_of_week_name STRING,
	is_weekend BOOLEAN,
	is_holiday BOOLEAN,
	loaded_at TIMESTAMP
) USING DELTA;

In [0]:
%sql
DECLARE OR REPLACE VARIABLE CALENDAR_YEAR INT DEFAULT 2025;

CREATE TABLE IF NOT EXISTS sandbox.ingress.dim_calendar (
	calendar_date DATE,
	calendar_year INT,
	fiscal_year INT,
	fiscal_quarter INT,
	month_number INT,
	month_name STRING,
	day_of_month INT,
	day_of_week_number INT,
	day_of_week_name STRING,
	is_weekend BOOLEAN,
	is_holiday BOOLEAN,
	loaded_at TIMESTAMP
) USING DELTA;

INSERT INTO sandbox.ingress.dim_calendar
SELECT
	calendar_date,
	year(calendar_date) AS calendar_year,
	year(calendar_date) AS fiscal_year,
	quarter(calendar_date) AS fiscal_quarter,
	month(calendar_date) AS month_number,
	date_format(calendar_date, 'MMMM') AS month_name,
	day(calendar_date) AS day_of_month,
	dayofweek(calendar_date) AS day_of_week_number,
	date_format(calendar_date, 'EEEE') AS day_of_week_name,
	dayofweek(calendar_date) IN (1, 7) AS is_weekend,
	false AS is_holiday,
	current_timestamp() AS loaded_at
FROM (
	SELECT explode(
		sequence(
			make_date(calendar_year, 1, 1),
			make_date(calendar_year, 12, 31),
			interval 1 day
		)
	) AS calendar_date
);


---

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

In [0]:
ing_cal_df = DeltaTable.forName(spark,"sandbox.ingress.dim_calendar").toDF()
brz_cal = DeltaTable.forName(spark,"sandbox.bronze.dim_calendar")

brz_cal.alias("brz").merge(
    ing_cal_df.alias("ing"),
    col("brz.calendar_date") == col("ing.calendar_date")
).whenNotMatchedInsertAll().execute()

___

In [0]:
DeltaTable.forName(spark, "sandbox.bronze.dim_calendar").toDF().count()

In [0]:
%sql
TRUNCATE TABLE sandbox.bronze.dim_calendar; 